In [ ]:
from rdkit.Chem import rdDepictor
from collections import namedtuple
import re

def make_rect(match):
    cx = match.group(1)
    cy = match.group(2)
    rx = float(match.group(3))
    ry = float(match.group(4))
    atom_class = match.group(5)
    style_str = match.group(6)
    width = rx * 1.5
    height = ry * 1.5
    x = float(cx) - (width / 2)
    y = float(cy) - (height / 2)

    return f"<rect x='{x}' y='{y}' width='{width}' height='{height}' rx='3' class='{atom_class}' style='{style_str}' />"

def hex_replace(match):
    full_style = match.group(0)
    # Find the hex code inside the style string
    hex8 = re.search(r'#([0-9A-Fa-f]{6})([0-9A-Fa-f]{2})', full_style)
    if hex8:
        color, alpha = hex8.groups()
        opacity = round(int(alpha, 16) / 255.0, 3)
        # Replace 8-digit with 6-digit
        fixed = full_style.replace(f"#{color}{alpha}", f"#{color}")
        # Add opacity attributes before the closing quote
        return fixed[:-1] + f";fill-opacity:{opacity};stroke-opacity:{opacity}'"
    return full_style


def replace_circles_with_rectangles(svg):
    pattern = r"<ellipse\s+cx='([^']+)'\s+cy='([^']+)'\s+rx='([^']+)'\s+ry='([^']+)'\s+class='([^']+)'\s+style='([^']+)'\s*/>"

    svg = re.sub(pattern, make_rect, svg)
    return re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, svg)


def DrawMorganBits(tpls, **kwargs):
  envs = []
  for tpl in tpls:
    if len(tpl) == 4:
      mol, bitId, bitInfo, whichExample = tpl
    else:
      mol, bitId, bitInfo = tpl
      whichExample = 0

    atomId, radius = bitInfo[bitId][whichExample]
    envs.append((mol, atomId, radius))
  return DrawMorganEnvs(envs, **kwargs)


# adapted from the function drawFPBits._drawFPBit() from the CheTo package
# original author Nadine Schneider
FingerprintEnv = namedtuple(
  'FingerprintEnv',
  ('submol', 'highlightAtoms', 'atomColors', 'highlightBonds', 'bondColors', 'highlightRadii'))


def _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor, extraColor,
                  **kwargs):
  if not mol.GetNumConformers():
    rdDepictor.Compute2DCoords(mol)
  bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atomId)

  # get the atoms for highlighting
  atomsToUse = set((atomId, ))
  for b in bitPath:
    atomsToUse.add(mol.GetBondWithIdx(b).GetBeginAtomIdx())
    atomsToUse.add(mol.GetBondWithIdx(b).GetEndAtomIdx())

  #  enlarge the environment by one further bond
  enlargedEnv = set()
  for atom in atomsToUse:
    a = mol.GetAtomWithIdx(atom)
    for b in a.GetBonds():
      bidx = b.GetIdx()
      if bidx not in bitPath:
        enlargedEnv.add(bidx)
  enlargedEnv = list(enlargedEnv)
  enlargedEnv += bitPath

  # set the coordinates of the submol based on the coordinates of the original molecule
  amap = {}
  if enlargedEnv:
    submol = Chem.PathToSubmol(mol, enlargedEnv, atomMap=amap)
  else:
    # generate submol from fragments with no bonds
    submol = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(mol, atomsToUse=atomsToUse))
  Chem.FastFindRings(submol)
  conf = Chem.Conformer(submol.GetNumAtoms())
  confOri = mol.GetConformer(0)
  for i1, i2 in amap.items():
    conf.SetAtomPosition(i2, confOri.GetAtomPosition(i1))
  submol.AddConformer(conf)

  for orig_idx, sub_idx in amap.items():
        sub_atom = submol.GetAtomWithIdx(sub_idx)

        # Check if the atom belongs to the "Core" bit
        if orig_idx in atomsToUse:
            orig_atom = mol.GetAtomWithIdx(orig_idx)
            h_count = orig_atom.GetTotalNumHs()
            degree = orig_atom.GetDegree()
            sub_atom.SetProp("atomNote", f"H:{h_count},D:{degree}")
        else:
            # For atoms outside the core, ensure no label is set
            if sub_atom.HasProp("atomNote"):
                sub_atom.ClearProp("atomNote")
            sub_atom.SetAtomicNum(0)
            sub_atom.SetIsAromatic(False)

  envSubmol = []
  for b in bitPath:
      beginAtom = amap[mol.GetBondWithIdx(b).GetBeginAtomIdx()]
      endAtom = amap[mol.GetBondWithIdx(b).GetEndAtomIdx()]
      envSubmol.append(submol.GetBondBetweenAtoms(beginAtom, endAtom).GetIdx())

  # color all atoms of the submol in gray which are not part of the bit
  # highlight atoms which are in rings
  atomcolors, bondcolors = {}, {}
  highlightAtoms, highlightBonds = [], []
  highlightRadii = {}
  for aidx in amap.keys():
    if aidx in atomsToUse:
      color = None
      if centerColor and aidx == atomId:
        color = centerColor
      if color is not None:
        atomcolors[amap[aidx]] = color
        highlightAtoms.append(amap[aidx])
        highlightRadii[amap[aidx]] = baseRad
    else:
      #drawopt.atomLabels[amap[aidx]] = '*'
      submol.GetAtomWithIdx(amap[aidx]).SetAtomicNum(0)
      submol.GetAtomWithIdx(amap[aidx]).UpdatePropertyCache()
  color = extraColor
  for bid in submol.GetBonds():
    bidx = bid.GetIdx()
    if bidx not in envSubmol:
      bondcolors[bidx] = color
      bid.SetBondType(Chem.BondType.OTHER)
      bid.SetIsAromatic(False)

      bondcolors[bidx] = color
      highlightBonds.append(bidx)

  return FingerprintEnv(submol, highlightAtoms, atomcolors, highlightBonds, bondcolors,
                        highlightRadii)


def DrawMorganEnvs(envs, molsPerRow=3, subImgSize=(150, 150), baseRad=0.4, useSVG=True,
                   aromaticColor=(0.9, 0.9, 0.2), ringColor=(0.8, 0.8, 0.8),
                   centerColor=(0.788, 0, 0.329, 0.4), extraColor=(0.9, 0.9, 0.9), legends=None,
                   drawOptions=None, **kwargs):
  submols = []
  highlightAtoms = []
  atomColors = []
  highlightBonds = []
  bondColors = []
  highlightRadii = []
  for mol, atomId, radius in envs:
    menv = _getMorganEnv(mol, atomId, radius, baseRad, aromaticColor, ringColor, centerColor,
                         extraColor, **kwargs)
    submols.append(menv.submol)
    highlightAtoms.append(menv.highlightAtoms)
    atomColors.append(menv.atomColors)
    highlightBonds.append(menv.highlightBonds)
    bondColors.append(menv.bondColors)
    highlightRadii.append(menv.highlightRadii)

  if legends is None:
    legends = [''] * len(envs)

  nRows = len(envs) // molsPerRow
  if len(envs) % molsPerRow:
    nRows += 1

  fullSize = (molsPerRow * subImgSize[0], nRows * subImgSize[1])
  # Drawing
  if useSVG:
    drawer = rdMolDraw2D.MolDraw2DSVG(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])
  else:
    drawer = rdMolDraw2D.MolDraw2DCairo(fullSize[0], fullSize[1], subImgSize[0], subImgSize[1])

  if drawOptions is None:
    drawOptions = drawer.drawOptions()
  drawOptions.prepareMolsBeforeDrawing = False
  drawOptions.continuousHighlight = False
  drawOptions.includeMetadata = False
  drawer.SetDrawOptions(drawOptions)
  drawer.DrawMolecules(submols, legends=legends, highlightAtoms=highlightAtoms,
                       highlightAtomColors=atomColors, highlightBonds=highlightBonds,
                       highlightBondColors=bondColors, highlightAtomRadii=highlightRadii, **kwargs)
  drawer.FinishDrawing()
  svg = drawer.GetDrawingText()

  return replace_circles_with_rectangles(svg)

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, SVG

def summarize_bits(smiles_list, target_bit, radius=2, n_bits=2048, use_chiral=True):
    """
    Groups molecules by their core bit structure and visualizes unique cores.
    """
    unique_cores = {}

    for m_idx, smiles in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smiles)
        if not mol: continue

        Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
        bit_info = {}
        AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits, bitInfo=bit_info, useChirality=use_chiral)

        if target_bit in bit_info:
            # Get the core substructure for aggregation
            atomId, rad = bit_info[target_bit][0]
            bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atomId)

            # Extract just the core (black part) to generate a grouping key
            core_submol = Chem.PathToSubmol(mol, bitPath)
            core_key = Chem.MolToSmiles(core_submol, isomericSmiles=use_chiral)

            if core_key not in unique_cores:
                unique_cores[core_key] = {
                    'mol': mol,
                    'atomId': atomId,
                    'rad': rad,
                    'mol_indices': set()
                }
            unique_cores[core_key]['mol_indices'].add(m_idx)

    # Prepare data for DrawMorganEnvs
    envs_to_draw = []
    legends = []
    for core_key, data in unique_cores.items():
        envs_to_draw.append((data['mol'], data['atomId'], data['rad']))
        legends.append(f"Found in {len(data['mol_indices'])} molecules")

    if not envs_to_draw:
        return "No occurrences found."

    img = DrawMorganEnvs(envs_to_draw, legends=legends, subImgSize=(300, 300))
    return img



def visualize_target_bit(smiles: list, target_bit: int, radius: int = 2, n_bits: int = 2048, use_chiral: bool = True):
    """
    Visualizes a specific Bit ID using RDKit's native DrawMorganBits.
    """
    # 1. Prepare Molecule
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        print("Invalid SMILES")
        return

    # Assign Stereochemistry so RDKit knows about R/S labels
    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)

    # 2. Calculate Fingerprint with Chirality
    bit_info = {}
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=n_bits,
        bitInfo=bit_info,
        useChirality=use_chiral
    )

    # 3. Validation: Check if the user's bit exists
    if target_bit not in bit_info:
        return

    # 4. Prepare Data for DrawMorganBits
    # bit_info[bit_id] is a list of tuples: (atom_index, radius)
    # We want to visualize all instances (e.g. if the bit appears twice)
    instances = bit_info[target_bit]

    tuples_to_draw = []
    legends = []

    print(f"✅ Bit {target_bit} found! Occurrences: {len(instances)}")
    unique_substructures = []
    for i, (atom_idx, rad) in enumerate(instances):
        bitPath = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
        core_submol = Chem.PathToSubmol(mol, bitPath)
        core_key = Chem.MolToSmiles(core_submol, isomericSmiles=use_chiral)

        if core_key in unique_substructures:
            continue  # Skip duplicate substructures
        unique_substructures.append(core_key)
        tuples_to_draw.append((mol, target_bit, bit_info))

        # Create a descriptive legend
        atom_symbol = mol.GetAtomWithIdx(atom_idx).GetSymbol()

        # Check for chirality info on the central atom
        atom_obj = mol.GetAtomWithIdx(atom_idx)
        cip = atom_obj.GetProp('_CIPCode') if atom_obj.HasProp('_CIPCode') else ""
        stereo_str = f"({cip})" if cip else ""

        legends.append(f"Atom: {atom_symbol}{stereo_str} (Idx {atom_idx}) | R: {rad}")

    # 5. Draw
    img = Draw.DrawMorganBits(
        tuples_to_draw,
        molsPerRow=min(len(instances), 4), # Adjust grid size
        subImgSize=(500, 500),
        legends=legends,
        useSVG=True
    )

    return img

def visualize_on_molecule(smiles: str, i, target_bit: int, radius: int = 2, n_bits: int = 2048, use_chiral: bool = True):
    """
    Visualizes a specific Bit ID highlighted on the molecule.
    """
    # 1. Prepare Molecule
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        mol = Chem.MolFromSmarts(smiles)
        print(mol)
        if not mol:
            print("Invalid SMILESaaa")
            return

    # Assign Stereochemistry so RDKit knows about R/S labels
    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)

    # 2. Calculate Fingerprint with Chirality
    bit_info = {}
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=radius, nBits=n_bits, bitInfo=bit_info, useChirality=use_chiral
    )
    print(bit_info)

    if target_bit not in bit_info:
        return None

    instances = bit_info[target_bit]

    # 1. Track highlights and specific colors
    highlight_atoms = []
    highlight_bonds = []
    atom_colors = {}  # Dictionary to map atom index to color (R, G, B)

    # Define our Blue color for the central atoms
    red_color = (0.204, 1, 0.941, 0.4)
    blue_color = (0.788, 0, 0.329, 0.4)

    highlight_atoms = []
    highlight_bonds = []
    atom_colors = {}
    bond_colors = {}

    instances = bit_info[target_bit]

    for atom_idx, rad in instances:
        # Mark center as Blue
        atom_colors[atom_idx] = blue_color
        highlight_atoms.append(atom_idx)

        if rad > 0:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, rad, atom_idx)
            for b_idx in env:
                # FORCE BONDS TO BE RED
                bond_colors[b_idx] = red_color
                highlight_bonds.append(b_idx)

                # Get atoms connected to this bond
                bond = mol.GetBondWithIdx(b_idx)
                aid1, aid2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()

                # If these atoms aren't the center, make them red
                for aid in [aid1, aid2]:
                    highlight_atoms.append(aid)
                    if aid not in atom_colors: # Don't overwrite the blue center
                        atom_colors[aid] = red_color

    # 3. Draw
    drawer = rdMolDraw2D.MolDraw2DSVG(600, 600)

    # This prevents the "bleeding" of colors
    drawer.drawOptions().prepareMolsBeforeDrawing = True

    drawer.drawOptions().highlightRadius = 0.4
    drawer.DrawMolecule(
        mol,
        highlightAtoms=list(set(highlight_atoms)),
        highlightAtomColors=atom_colors,
        highlightBonds=list(set(highlight_bonds)),
        highlightBondColors=bond_colors,
        legend=f'{i}'
    )
    drawer.FinishDrawing()
    img = drawer.GetDrawingText()

    return replace_circles_with_rectangles(img)

In [ ]:
import os
import pandas as pd

smiles_path = '../../../data/predictor/cof_matched/rth_0.99/features/features_rth_0.99_estate_sum.csv'
df = pd.read_csv(smiles_path)
smiles_lists = df['smiles'].tolist()

In [ ]:
RADIUS = 2
N_BITS = 1024
USE_CHIRALITY = True
TARGET_BIT_ID = 116 #[111, 684, 122, 884, 43, 942, 338, 18, 675]
OUTPUT_FOLDER = "bit_visualizations_NEW_FINAL"

In [ ]:
import warnings
warnings.simplefilter("ignore") # pearl aqua

#smiles_lists should be your list of SMILES strings
# for s in smiles_lists:
#     img = visualize_target_bit(s, TARGET_BIT_ID, radius=RADIUS, n_bits=N_BITS,
#                          use_chiral=USE_CHIRALITY)
#     display(img)
os.makedirs(f'{OUTPUT_FOLDER}/ECFP{TARGET_BIT_ID}', exist_ok=True)

for i in [116, 875, 830, 338, 675, 136, 849, 122]:
    img = summarize_bits(smiles_lists, i, radius=RADIUS, n_bits=N_BITS,
                   use_chiral=USE_CHIRALITY)
    os.makedirs(f'{OUTPUT_FOLDER}/ECFP{i}', exist_ok=True)
    with open(f'{OUTPUT_FOLDER}/ECFP{i}/all_types.svg', 'w') as f:
            f.write(img)
    # print(f"Saved: {filename}")
    display(SVG(img))

In [ ]:

TARGET_BIT_ID = 849
j = 0

for i in [6]: #range(len(smiles_lists)):
    s = smiles_lists[i]
    # img = visualize_on_molecule(s, i, TARGET_BIT_ID, radius=RADIUS, n_bits=N_BITS,
    #                      use_chiral=USE_CHIRALITY)
    img = visualize_on_molecule(s, i, TARGET_BIT_ID, radius=RADIUS, n_bits=N_BITS,
                         use_chiral=USE_CHIRALITY)
    # if img is not None:
    #     with open(f'smiles_{i}/ecfp_{TARGET_BIT_ID}.svg', 'w') as f:
    #         f.write(img)
    # os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_FOLDER, f"ECFP{TARGET_BIT_ID}"), exist_ok=True)
    if img is not None:
        with open(f'{OUTPUT_FOLDER}/ECFP{TARGET_BIT_ID}/example_{i}.svg', 'w') as f:
             f.write(img)

        display(SVG(img))
        j += 1
    if j >= 45:  # Limit to first 5 for demonstration
        break

In [ ]:
from rdkit.Chem.EState.AtomTypes import _rawD

feature_names = [name for name, smarts in _rawD]
dicts = {i: smarts for i, (name, smarts) in enumerate(_rawD)}
print(dicts[17])

In [ ]:
patterns = {
    20: "[ND1H2]-*", #sNH2
    31: "[N,nD3H0](:*)(:*)-,:*",
    # 11: aaCH (Aromatic Carbon with one Hydrogen)
    # The ':' indicates aromatic bonds
    11: "[C,c;D2H](:*):*", #aaCH

    # 27: sN (Nitrogen with a double bond, e.g., Imine or Pyridine)
    27: "[ND2H0](=*)-*", #dsN
    33: "[OD1H]-*", #sOH
    16: "[C,c;D3H0](:*)(:*)-*", #aasC
    28: "[N,nD2H0](:*):*", #aaN
    17: "[C,c;D3H0](:*)(:*):*", #aaaC
    34: "[OD1H0]=*", #dO
    50: "[S,sD2H0](:*):*",
    15: "[CD3H0](=*)(-*)-*"
}

def _label_query_atom(atom):
    atom_smarts = atom.GetSmarts()

    # Hide connector wildcards, e.g. '*' in (:*)
    if atom.GetAtomicNum() == 0 and atom_smarts in {"*", "[*]"}:
        return " "

    # Keep complex SMARTS query atoms visible, but as compact element labels.
    if atom.GetAtomicNum() == 0:
        inner = atom_smarts.strip("[]")
        first_choice = inner.split(",", 1)[0]
        match = re.search(r"[A-Z][a-z]?|[a-z]", first_choice)
        if match:
            return match.group(0).upper()
        return "?"

    label = atom.GetSymbol()
    num_h = atom.GetNumExplicitHs()
    if num_h > 0:
        label += "H" + (f"<sub>{num_h}</sub>" if num_h > 1 else "")
    return label


def _build_colored_display_mol(query_mol):
    # Query atoms (atomic number 0) are always black; draw a plain molecule copy for colorized labels.
    pt = Chem.GetPeriodicTable()
    rw_mol = Chem.RWMol()
    labels = []

    for atom in query_mol.GetAtoms():
        label = _label_query_atom(atom)
        labels.append(label)

        elem_match = re.match(r"[A-Z][a-z]?", label) if label.strip() else None
        atomic_num = pt.GetAtomicNumber(elem_match.group(0)) if elem_match else 0
        rw_mol.AddAtom(Chem.Atom(atomic_num))

    for bond in query_mol.GetBonds():
        rw_mol.AddBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond.GetBondType())

    display_mol = rw_mol.GetMol()
    display_mol.UpdatePropertyCache(strict=False)

    for idx, label in enumerate(labels):
        display_mol.GetAtomWithIdx(idx).SetProp("atomLabel", label)

    rdDepictor.Compute2DCoords(display_mol)
    return display_mol


def visualize_estate_substructure(id):
    feature_names = [name for name, smarts in _rawD]
    dict = {i: smarts for i, (name, smarts) in enumerate(_rawD)}
    print(dict[id])
    smarts = patterns[id]
    query_mol = Chem.MolFromSmarts(smarts)

    query_mol.UpdatePropertyCache(strict=False)
    mol = _build_colored_display_mol(query_mol)

    drawer = rdMolDraw2D.MolDraw2DSVG(200, 300)
    options = drawer.drawOptions()
    options.useComplexQueryAtomSymbols = True
    options.useDefaultAtomPalette()
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    svg_text = drawer.GetDrawingText()
    return svg_text


for i in [34, 11]:
    img = visualize_estate_substructure(i)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_FOLDER, f"EState{i}"), exist_ok=True)
    with open(f'{OUTPUT_FOLDER}/EState{i}/sub.svg', 'w') as f:
        f.write(img)
    display(SVG(img))
img = visualize_estate_substructure(20)
display(SVG(img))

In [ ]:
from rdkit.Geometry import Point2D
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import EState
from rdkit.Chem.Draw import SimilarityMaps
import io
import cairosvg
from IPython.display import Image
def estate_types():
    from rdkit.Chem.EState.AtomTypes import _rawD

    feature_names = [name for name, smarts in _rawD]
    return np.asarray(feature_names, dtype=object)
ESTATE_TYPES = estate_types()

def visualize_estate_heatmap(smiles: str, idd, feature_id: int, all_smiles: list = None):

    mol = Chem.MolFromSmiles(smiles)

    if not mol: return None
    AllChem.Compute2DCoords(mol)
    conformer = mol.GetConformer()

    # 1. Calculate Full EState Indices (Intrinsic + Delta) for all atoms
    estate_indices = EState.EStateIndices(mol)

    target_type_name = ESTATE_TYPES[feature_id]

    max_weight = 0
    for s in all_smiles:
        estate = EState.EStateIndices(Chem.MolFromSmiles(s))
        atom_types = EState.AtomTypes.TypeAtoms(Chem.MolFromSmiles(s))
        atom_types = [t[0] if len(t) > 0 else '' for t in atom_types ]
        for i, a_type in enumerate(atom_types):
            if a_type == target_type_name:
                if abs(estate[i]) > max_weight:
                    max_weight = abs(estate[i])

    # 2. Get the Kier-Hall type string for every atom in the molecule
    # This returns a list of strings like ('sCH3', 'ssCH2', etc.)
    atom_types = EState.AtomTypes.TypeAtoms(mol)
    atom_types = [t[0] if len(t) > 0 else '' for t in atom_types ]  # Extract just the type names

    # 3. Identify the target type name from the feature_id
    if not (0 <= feature_id < len(ESTATE_TYPES)):
        print(f"Invalid ID: {feature_id}. Use 0-78.")
        return None


    # 4. Filter weights: only atoms matching the requested EState type get a value
    weights = []
    matching_atoms = [] # To store which atoms we want to label
    for i, a_type in enumerate(atom_types):
        if a_type == target_type_name:
            weights.append(float(estate_indices[i]))
            matching_atoms.append(i)
        else:
            weights.append(0.0)

    sum_weights = round(sum(weights), 2)
    if not matching_atoms:
        return None

    drawer = rdMolDraw2D.MolDraw2DSVG(700, 700)

    if max_weight > 0:
        h_weights = [w / max_weight for w in weights]
    else:
        h_weights = weights

    SimilarityMaps.GetSimilarityMapFromWeights(
        mol,
        h_weights,
        draw2d=drawer,
        step=0.01,
        sigma=0.5,
    )

    for idx in matching_atoms:
        # Get atom position in Angstroms
        pos = conformer.GetAtomPosition(idx)
        atom_pos = Point2D(pos.x + 1, pos.y + 1)
        # Get the raw EState value
        val_text = f"{estate_indices[idx]:.2f}"
        # Draw the value slightly offset from the atom center
        # We use rawCoords=True because atom positions are in Angstroms
        drawer.SetFontSize(15)
        drawer.DrawString(val_text, atom_pos, rawCoords=False)

    # text_pos = Point2D(50, 50)
    # drawer.SetFontSize(12)
    #
    # drawer.DrawString(
    #     f"{idd}, sum: {sum_weights}",
    #     text_pos,
    #     rawCoords=True
    # )

    drawer.FinishDrawing()
    svg_text = drawer.GetDrawingText()

    return svg_text

# --- Usage Example ---
feature = 28
j = 0
for i in [1]: #range(len(smiles_lists)):
    s = smiles_lists[i]
    print(i, s)
    img = visualize_estate_heatmap(s, i, feature, smiles_lists)
    if img is not None:
        with open(f'{OUTPUT_FOLDER}/EState{feature}/example_{i}.svg', 'w') as f:
            img = re.sub(r"style='[^']+#([0-9A-Fa-f]{8})[^']*'", hex_replace, img)
            f.write(img)
        display(SVG(img))
        j += 1
    if j >= 15:  # Limit to first 5 for demonstration
        break
# smiles = "CC1=CC=C(C)C=C1" # Para-xylene
# feature = 10 # aaCH (Aromatic carbons)
# img = visualize_estate_heatmap(smiles, feature)
# if img: display(img)

In [ ]:
def visualize_fg_substructure(substructure):
    mol = Chem.MolFromSmarts(substructure)

    mol.UpdatePropertyCache(strict=False)

    # 2. MANUALLY SET LABELS (The Secret Sauce)
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        num_h = atom.GetNumExplicitHs() # This reads the '2' in [NH2]

        if symbol == "*":
            # Make dummy atoms invisible so we focus on the group
            atom.SetProp("atomLabel", " ")
        elif symbol == "C":
            # Use RDKit default carbon rendering (implicit C labels).
            if atom.HasProp("atomLabel"):
                atom.ClearProp("atomLabel")
        else:
            # Build the label: Symbol + H + count
            label = symbol
            if num_h > 0:
                h_text = "H" + (f"<sub>{num_h}</sub>" if num_h > 1 else "")
                label += h_text
            atom.SetProp("atomLabel", label)

    drawer = rdMolDraw2D.MolDraw2DSVG(200, 300)
    options = drawer.drawOptions()
    options.useComplexQueryAtomSymbols = False
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    svg_text = drawer.GetDrawingText()
    return svg_text



fg = {
    'benzene': ['c1ccccc1', 'c1ccccc1'],
    'bicyclic': ['[R2][R2]', '[R2][R2]'],
    'tr_amides': ['[n;H0]','[n;H0]'],
    'allylic1': ['C=C-C', '[$(C=C-C);!$(C=C-C-[N,O,S]);!$(C=C-C-C-[N,O]);!$(C12=CC(=O)CCC1C3C(C4C(CCC4)CC3)CC2)]']
}


for name, pattern in fg.items():
    img = visualize_fg_substructure(pattern[0])
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_FOLDER, f"FG_{name}"), exist_ok=True)
    with open(f'{OUTPUT_FOLDER}/FG_{name}/sub.svg', 'w') as f:
        f.write(img)
    display(SVG(img))

In [ ]:
def _extract_positive_recursive_smarts(smarts: str):
    """Return positive recursive SMARTS terms from patterns like [$(A);!$(B)]."""
    terms = []
    i = 0
    while i < len(smarts):
        if smarts[i] == '$' and i + 1 < len(smarts) and smarts[i + 1] == '(':
            negated = i > 0 and smarts[i - 1] == '!'
            depth = 1
            j = i + 2
            while j < len(smarts) and depth > 0:
                if smarts[j] == '(':
                    depth += 1
                elif smarts[j] == ')':
                    depth -= 1
                j += 1
            if depth == 0 and not negated:
                terms.append(smarts[i + 2:j - 1])
            i = j
            continue
        i += 1
    return terms


def visualize_fg_on_molecule(smiles: str, i, functional_group_smarts: str, highlight_core_smarts: str = None):
    """
    Highlights all occurrences of a functional-group SMARTS on a molecule.
    For recursive SMARTS, use full SMARTS for filtering and a core SMARTS for bond-level highlighting.
    """
    mol = Chem.MolFromSmiles(smiles)
    if not mol:
        print("Invalid SMILES")
        return None

    filter_query = Chem.MolFromSmarts(functional_group_smarts)
    if not filter_query:
        print("Invalid functional-group SMARTS")
        return None

    # If no explicit core is provided, try to recover the first positive recursive clause.
    if highlight_core_smarts is None:
        positive_terms = _extract_positive_recursive_smarts(functional_group_smarts)
        if positive_terms:
            highlight_core_smarts = positive_terms[0]

    core_query = Chem.MolFromSmarts(highlight_core_smarts) if highlight_core_smarts else filter_query

    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
    filter_matches = mol.GetSubstructMatches(filter_query, uniquify=True)
    if not filter_matches:
        return None

    anchor_atoms = {m[0] for m in filter_matches if len(m) > 0}

    blue_color = (0.204, 1, 0.941, 0.4)
    highlight_atoms = set()
    highlight_bonds = set()
    atom_colors = {}
    bond_colors = {}

    # Preferred path: highlight the full core motif and keep only matches accepted by the filter query.
    if core_query is not None and core_query.GetNumAtoms() > 0:
        core_matches = mol.GetSubstructMatches(core_query, uniquify=True)

        # If filter is single-atom recursive SMARTS, map accepted anchors to core matches by first atom.
        if filter_query.GetNumAtoms() == 1 and core_query.GetNumAtoms() > 1:
            core_matches = [m for m in core_matches if m and m[0] in anchor_atoms]

        for match in core_matches:
            for atom_idx in match:
                highlight_atoms.add(atom_idx)
                atom_colors[atom_idx] = blue_color

            for qbond in core_query.GetBonds():
                beg = match[qbond.GetBeginAtomIdx()]
                end = match[qbond.GetEndAtomIdx()]
                mbond = mol.GetBondBetweenAtoms(beg, end)
                if mbond is None:
                    continue
                bidx = mbond.GetIdx()
                highlight_bonds.add(bidx)
                bond_colors[bidx] = blue_color

    # Fallback: at least highlight filtered atoms.
    if not highlight_atoms:
        for match in filter_matches:
            for atom_idx in match:
                highlight_atoms.add(atom_idx)
                atom_colors[atom_idx] = blue_color

    drawer = rdMolDraw2D.MolDraw2DSVG(600, 600)
    drawer.drawOptions().prepareMolsBeforeDrawing = True
    drawer.drawOptions().highlightRadius = 0.4
    drawer.DrawMolecule(
        mol,
        highlightAtoms=sorted(highlight_atoms),
        highlightAtomColors=atom_colors,
        highlightBonds=sorted(highlight_bonds),
        highlightBondColors=bond_colors,
        legend=f"{i}"
    )
    drawer.FinishDrawing()
    img = drawer.GetDrawingText()

    return replace_circles_with_rectangles(img)

In [ ]:
j = 0
fg = {
    'benzene': ['c1ccccc1', 'c1ccccc1'],
    'bicyclic': ['[R2][R2]', '[R2][R2]'],
    'tr_amides': ['[#7]','[#7]'],
    'allylic1': ['C=C-C', '[$(C=C-C);!$(C=C-C-[N,O,S]);!$(C=C-C-C-[N,O]);!$(C12=CC(=O)CCC1C3C(C4C(CCC4)CC3)CC2)]'],
    'imidazone': ['n1cncc1	n1cncc1', 'n1cncc1	n1cncc1']
}
FUNC_GROUP = 'benzene'
# smiles_lists = []
for i in [1]: #range(len(smiles_lists)):
    s = smiles_lists[i]
    img = visualize_fg_on_molecule(s, i, fg[FUNC_GROUP][1], highlight_core_smarts=fg[FUNC_GROUP][0])

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_FOLDER, f"FG_{FUNC_GROUP}"), exist_ok=True)

    if img is not None:
        j += 1
        with open(f'{OUTPUT_FOLDER}/FG_{FUNC_GROUP}/example_{i}.svg', 'w') as f:
             f.write(img)
        display(SVG(img))
    if j > 15:
        break

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw, rdmolops, GetDistanceMatrix
from IPython.display import display # Required to show images in a loop in Notebooks

# 1. Helper function to calculate diameter and get path indices
def get_diameter_path(mol: Chem.Mol):
    # Calculate Distance Matrix
    dm = GetDistanceMatrix(mol)

    # Get diameter value
    diameter_val = int(np.max(dm))

    # Find start and end atoms (indices of the max value in the matrix)
    flat_idx = np.argmax(dm)
    source_idx, target_idx = np.unravel_index(flat_idx, dm.shape)

    # Get the path atoms
    path_atoms = list(rdmolops.GetShortestPath(mol, int(source_idx), int(target_idx)))

    # Get the path bonds (for nice continuous lines)
    path_bonds = []
    for i in range(len(path_atoms) - 1):
        bond = mol.GetBondBetweenAtoms(path_atoms[i], path_atoms[i+1])
        if bond:
            path_bonds.append(bond.GetIdx())

    return diameter_val, path_atoms, path_bonds

def save_diameter_svg(mol, filename, width=300, height=300):
    d_val, atoms, bonds = get_diameter_path(mol)

    # Initialize the SVG drawer
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)

    opts = drawer.drawOptions()
    opts.addStereoAnnotation = False

    # Draw the molecule
    drawer.DrawMolecule(
        mol,
        highlightAtoms=atoms,
        highlightBonds=bonds,
        legend=f"Diameter: {d_val}"
    )

    # Finalize and get the SVG string
    drawer.FinishDrawing()
    svg_string = drawer.GetDrawingText()
    return svg_string

    # Write to file
    # with open(filename, 'w') as f:
    #     f.write(svg_string)
    # print(f"Saved: {filename}")

dataset = [Chem.MolFromSmiles(s) for s in smiles_lists]

ids = [0, 2, 4, 8, 9, 31]
smiles_ids = [smiles_lists[i] for i in ids]

names = [smiles_lists[1], smiles_lists[15]]
imgs = []

for i, mol in enumerate(smiles_ids):
    m = Chem.MolFromSmiles(mol)
    print(i, mol, get_diameter_path(m)[0])
    img = save_diameter_svg(m, f'diameter_{i}.svg', width=300, height=300)
    imgs.append(img)


In [ ]:
import math
import cairosvg
import io
from PIL import Image

def save_svg_list_as_png_grid(svg_list, output_filename, mols_per_row=3, dpi=300):
    if not svg_list:
        return

    # 1. Determine cell size from the first SVG
    w_match = re.search(r'width=[\'"](\d+)px[\'"]', svg_list[0])
    h_match = re.search(r'height=[\'"](\d+)px[\'"]', svg_list[0])
    cell_w = int(w_match.group(1)) if w_match else 300
    cell_h = int(h_match.group(1)) if h_match else 300

    # 2. Build the Master SVG string
    rows = math.ceil(len(svg_list) / mols_per_row)
    total_w = cell_w * mols_per_row
    total_h = cell_h * rows

    combined_svg = [
        f'<svg version="1.1" xmlns="http://www.w3.org/2000/svg" '
        f'width="{total_w}" height="{total_h}">'
    ]

    for i, svg_str in enumerate(svg_list):
        x = (i % mols_per_row) * cell_w
        y = (i // mols_per_row) * cell_h

        # Clean the string (remove XML headers)
        clean_svg = re.sub(r'<\?xml.*?\?>|<!DOCTYPE.*?>', '', svg_str)
        # Nest inside a new svg tag with coordinates
        combined_svg.append(f'<svg x="{x}" y="{y}">{clean_svg}</svg>')

    combined_svg.append('</svg>')
    full_svg_string = "".join(combined_svg)

    # 3. Convert the Master SVG to PNG
    # We use scale to increase the DPI/Quality if needed
    cairosvg.svg2png(bytestring=full_svg_string.encode('utf-8'),
                     write_to=output_filename,
                     scale=2.0) # scale=2.0 makes it sharper/double resolution

    print(f"Grid saved as PNG: {output_filename}")

